# Computer Vision Lab 01

This notebook implements four introductory OpenCV operations: selective red colour isolation, image blending, a soft vignette, and channel swapping. It reads the original images from `images/` and saves the results in `outputs/`. Run the cells from top to bottom.

**Requirements:** Python, OpenCV, NumPy, and Matplotlib. Google Colab includes the plotting tools; install OpenCV only if the import cell reports that it is missing.

## Prepare the notebook environment

When opened from GitHub in Colab, the notebook clones the course repository so the original images are available. If the `images/` folder is already present in the current runtime, it uses that folder directly.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_root = Path('/content/Computer-Vision-Course')
if not Path('images/apple.png').exists():
    if not (repo_root / 'LAB 01' / 'images' / 'apple.png').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/KerolusHany/Computer-Vision-Course.git', str(repo_root)],
            check=True
        )
    os.chdir(repo_root / 'LAB 01')

IMAGE_DIR = Path('images')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Working directory:', Path.cwd())
print('Input images:', sorted(p.name for p in IMAGE_DIR.iterdir()))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print('OpenCV:', cv2.__version__)

def read_image(filename):
    image = cv2.imread(str(IMAGE_DIR / filename), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f'Could not read {IMAGE_DIR / filename}')
    return image

def save_image(filename, image):
    path = OUTPUT_DIR / filename
    if not cv2.imwrite(str(path), image):
        raise OSError(f'Could not write {path}')
    print('Saved:', path)

def show_image(title, image, cmap=None):
    plt.figure(figsize=(8, 5))
    if cmap is None:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(image, cmap=cmap, vmin=0, vmax=255)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Task 1 Selective colour isolation

The image is converted to grayscale for the background. The mask keeps pixels whose red channel is sufficiently bright and exceeds both the blue and green channels by a margin. Bitwise operations combine the original colour pixels with the grayscale background.

In [ ]:
RED_MIN = 80
RED_DOMINANCE = 35

apple = read_image('apple.png')
blue, green, red = cv2.split(apple)
gray = cv2.cvtColor(apple, cv2.COLOR_BGR2GRAY)
gray_background = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
other_max = cv2.max(blue, green)
dominance = cv2.subtract(red, other_max)
mask = cv2.bitwise_and(
    cv2.threshold(red, RED_MIN, 255, cv2.THRESH_BINARY)[1],
    cv2.threshold(dominance, RED_DOMINANCE, 255, cv2.THRESH_BINARY)[1]
)
red_pixels = cv2.bitwise_and(apple, apple, mask=mask)
gray_pixels = cv2.bitwise_and(gray_background, gray_background, mask=cv2.bitwise_not(mask))
task1_result = cv2.bitwise_or(red_pixels, gray_pixels)
save_image('task1_selective_red.png', task1_result)
show_image('Task 1: Selective red colour', task1_result)

## Task 2 Ghostly overlay

The zebra image is resized to the cameraman image dimensions. `cv2.addWeighted` blends the two images using a 65% base-image weight and a 35% overlay weight.

In [ ]:
GHOST_ALPHA = 0.65
GHOST_BETA = 0.35

cameraman = read_image('cameraman.jpg')
zebra = read_image('zebra.png')
height, width = cameraman.shape[:2]
zebra_resized = cv2.resize(zebra, (width, height), interpolation=cv2.INTER_AREA)
task2_result = cv2.addWeighted(cameraman, GHOST_ALPHA, zebra_resized, GHOST_BETA, 0)
save_image('task2_ghost_overlay.png', task2_result)
show_image('Task 2: Ghostly overlay', task2_result)

## Task 3 Artistic vignette

A filled circle creates the bright centre of the vignette mask. Gaussian blur softens its edge, and the mask is multiplied with the Mars rover image to darken the outer region gradually.

In [ ]:
VIGNETTE_RADIUS_FRACTION = 0.42
VIGNETTE_BLUR_FRACTION = 0.12

rover = read_image('marsrover.png')
height, width = rover.shape[:2]
mask = np.zeros((height, width), dtype=np.uint8)
radius = int(min(height, width) * VIGNETTE_RADIUS_FRACTION)
cv2.circle(mask, (width // 2, height // 2), radius, 255, thickness=-1)
blur = max(1, int(min(height, width) * VIGNETTE_BLUR_FRACTION))
if blur % 2 == 0:
    blur += 1
soft_mask = cv2.GaussianBlur(mask, (blur, blur), 0)
task3_result = cv2.multiply(rover, cv2.cvtColor(soft_mask, cv2.COLOR_GRAY2BGR), scale=1 / 255)
save_image('task3_vignette.png', task3_result)
show_image('Task 3: Soft vignette', task3_result)

## Task 4 Channel swapping

OpenCV stores colour channels in BGR order. This task exchanges the green and red channel positions while leaving blue in place, producing a surreal colour effect.

In [ ]:
zebra = read_image('zebra.png')
blue, green, red = cv2.split(zebra)
task4_result = cv2.merge((blue, red, green))
save_image('task4_channel_swap.png', task4_result)
show_image('Task 4: Channel swap', task4_result)

## Results

The four processed images are saved as PNG files in `outputs/`. The report document in this folder describes the methods and results for submission.